# scvi/02 — Project Query (Pure Inference, No Retraining)

Loads the saved SCANVI reference model and projects new query cells by
passing them through the **frozen encoder** — no gradient updates, no
retraining epochs. This mirrors what Seurat CCA `MapQuery` does.

**API used:**
```python
scanvi_model = scvi.model.SCANVI.load(model_dir)          # load once
latent  = scanvi_model.get_latent_representation(adata_q)  # encoder forward pass
labels  = scanvi_model.predict(adata_q)                    # classifier forward pass
```

> **scArches (surgery) alternative:** for out-of-distribution data, brief fine-tuning
> via `SCANVI.load_query_data(adata_q, model)` + `model.train(max_epochs=100)` is more
> robust. Switch the `USE_SURGERY` flag below to compare.

**Inputs:** `data/scvi/query.h5ad`, `models/scanvi_reference/`  
**Outputs:** `data/scvi/query_projected.h5ad`

In [1]:
# Parameters — injected by papermill
query_h5ad            = "data/scvi/query.h5ad"
reference_h5ad_trained= "data/scvi/reference_trained.h5ad"
model_dir             = "models/scanvi_reference"
output_projected_h5ad = "data/scvi/query_projected.h5ad"
labels_key            = "cluster"
USE_SURGERY           = False  # True = scArches brief fine-tune; False = pure inference
n_epochs_surgery      = 100    # used only if USE_SURGERY=True


In [2]:
# Parameters
query_h5ad = "data/scvi/query.h5ad"
model_dir = "models/scanvi_reference"
output_projected_h5ad = "data/scvi/query_projected.h5ad"
USE_SURGERY = False


## 0 · Import

In [3]:
import os
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')

import anndata as ad
import scanpy as sc
import scvi
import os, time
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

scvi.settings.seed = 42
t_nb_start = time.time()
print(f"scvi-tools {scvi.__version__}")


Seed set to 42


scvi-tools 1.4.2


## 1 · Load Saved Model

In [4]:
print(f"Loading SCANVI model from {model_dir} ...")
t0 = time.time()
# save_anndata=True was used at save time, so reference AnnData is embedded
scanvi_model = scvi.model.SCANVI.load(model_dir)
t_load = time.time() - t0
print(f"  loaded in {t_load:.1f}s")
print(f"  gene set : {len(scanvi_model.adata.var_names):,} genes")
print(f"  labels   : {sorted(scanvi_model.adata.obs[labels_key].unique())}")


Loading SCANVI model from models/scanvi_reference ...
INFO     File models/scanvi_reference/model.pt already downloaded                                                  


/opt/anaconda3/envs/scrnaseq/lib/python3.11/site-packages/scvi/model/base/_base_model.py:862: UserWarning: `accelerator` has been automatically set to `cpu` although 'mps' exists. If you wish to run on mps backend, use explicitly accelerator='mps' in train function.In future releases it will become default for mps supported machines.
  _, _, device = parse_device_args(


  loaded in 0.2s
  gene set : 2,000 genes
  labels   : ['CD8_act', 'CD8_eff', 'CD8_ex', 'CD8_ex_act', 'CD8_mem', 'Naive', 'Tfh', 'Th17', 'Tregs']


## 2 · Load Query & Align Gene Set

Query must contain the same genes the model was trained on.
Extra query genes are dropped; genes missing from the query are zero-filled.

In [5]:
adata_query = ad.read_h5ad(query_h5ad)
print(f"Query raw: {adata_query.n_obs:,} cells x {adata_query.n_vars:,} genes")

# Align to reference gene set
ref_genes  = scanvi_model.adata.var_names
shared     = adata_query.var_names.intersection(ref_genes)
print(f"Shared genes with reference: {len(shared):,} / {len(ref_genes):,}")

if len(shared) < len(ref_genes):
    import scipy.sparse, numpy as np
    # Zero-fill missing genes
    import anndata as ad2
    missing = ref_genes.difference(adata_query.var_names)
    zero_block = scipy.sparse.csr_matrix((adata_query.n_obs, len(missing)))
    extra = ad.AnnData(
        X   = zero_block,
        obs = adata_query.obs,
        var = pd.DataFrame(index=missing)
    )
    adata_query = ad.concat([adata_query, extra], axis=1)

# Reorder to match reference exactly
adata_query = adata_query[:, ref_genes].copy()
# Ensure counts layer exists
if "counts" not in adata_query.layers:
    adata_query.layers["counts"] = adata_query.X.copy()
print(f"Query aligned: {adata_query.n_obs:,} cells x {adata_query.n_vars:,} genes")


Query raw: 30,683 cells x 17,865 genes
Shared genes with reference: 2,000 / 2,000


Query aligned: 30,683 cells x 2,000 genes


## 3 · Project Query

**Pure inference:** encoder and classifier weights are frozen.
**Surgery (optional):** brief fine-tune to adapt encoder to query distribution.

In [6]:
if USE_SURGERY:
    print("=== scArches surgery mode (brief fine-tune) ===")
    t_proj_start = time.time()
    surgery_model = scvi.model.SCANVI.load_query_data(
        adata_query, scanvi_model,
        freeze_expression=True,
    )
    surgery_model.train(
        max_epochs=n_epochs_surgery,
        plan_kwargs={"weight_decay": 0.0},
    )
    t_proj_elapsed = time.time() - t_proj_start
    active_model = surgery_model
    print(f"Surgery fine-tune: {t_proj_elapsed:.1f}s")
else:
    print("=== Pure inference mode (frozen model) ===")
    t_proj_start = time.time()
    active_model = scanvi_model  # use reference model directly
    t_proj_elapsed = 0.0

# Latent representation
t0 = time.time()
adata_query.obsm["X_scANVI"] = active_model.get_latent_representation(adata_query)
t_latent = time.time() - t0

# Label prediction
t0 = time.time()
adata_query.obs["predicted_label"] = active_model.predict(adata_query)
t_predict = time.time() - t0

print(f"get_latent_representation : {t_latent:.2f}s")
print(f"predict                   : {t_predict:.2f}s")
print(f"\nPredicted label distribution:")
print(adata_query.obs["predicted_label"].value_counts().to_string())


=== Pure inference mode (frozen model) ===
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             


get_latent_representation : 0.47s
predict                   : 0.38s

Predicted label distribution:
predicted_label
Th17          6753
CD8_mem       5435
Naive         4757
Tregs         4259
CD8_act       3761
Tfh           2834
CD8_ex        1514
CD8_eff        951
CD8_ex_act     419


## 4 · UMAP in scANVI Latent Space

In [7]:
sc.pp.neighbors(adata_query, use_rep="X_scANVI", n_neighbors=15)
sc.tl.umap(adata_query)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sc.pl.umap(adata_query, color="predicted_label",
           title="Query — scANVI predicted labels", ax=axes[0], show=False)
# Colour by ground-truth if available
gt_col = labels_key if labels_key in adata_query.obs.columns else "predicted_label"
sc.pl.umap(adata_query, color=gt_col,
           title=f"Query — {gt_col} (ground truth)", ax=axes[1], show=False)
plt.tight_layout()
plt.show()


OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


/var/folders/8y/ph922wl553g48t1wthjdsfnm0000gn/T/ipykernel_15652/3559250496.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5 · Identify CD8+ T Cells

In [8]:
# CD8 detection: check predicted labels; fall back to checking original metadata
cd8_mask = adata_query.obs["predicted_label"].str.contains(
    "CD8|cd8|exhausted|effector|TRM", case=False, na=False
)
adata_query.obs["is_cd8"] = cd8_mask
n_cd8 = cd8_mask.sum()
print(f"CD8+ T cells: {n_cd8:,} / {adata_query.n_obs:,} ({n_cd8/adata_query.n_obs*100:.1f}%)")

fig, ax = plt.subplots(figsize=(6, 5))
sc.pl.umap(adata_query,
           color="is_cd8", title=f"CD8+ T cells highlighted (n={n_cd8})",
           ax=ax, show=False)
plt.tight_layout()
plt.show()


CD8+ T cells: 12,080 / 30,683 (39.4%)


/var/folders/8y/ph922wl553g48t1wthjdsfnm0000gn/T/ipykernel_15652/310401472.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6 · Save Projected Query

In [9]:
os.makedirs(os.path.dirname(output_projected_h5ad), exist_ok=True)
adata_query.write_h5ad(output_projected_h5ad)
print(f"Saved -> {output_projected_h5ad}")
print(f"Key obs columns: {list(adata_query.obs.columns)}")


Saved -> data/scvi/query_projected.h5ad
Key obs columns: ['orig.ident', 'nCount_RNA', 'nFeature_RNA', 'patient', 'treatment', 'cluster', 'UMAP1', 'UMAP2', 'clonotype_id', 'has_tcr', 'percent.mt', 'RNA_snn_res.0.5', 'seurat_clusters', 'doublet_score', 'predicted_doublet', '_scvi_batch', '_scvi_labels', 'predicted_label', 'is_cd8']


## 7 · Timing Summary

In [10]:
t_total = time.time() - t_nb_start
mode_str = 'scArches surgery' if USE_SURGERY else 'pure inference'
print("=== scvi/02 Timing Summary ===")
print(f"  Mode                         : {mode_str}")
if USE_SURGERY:
    print(f"  Surgery fine-tune ({n_epochs_surgery} epochs): {t_proj_elapsed:7.1f}s  ({t_proj_elapsed/60:5.1f} min)")
print(f"  get_latent_representation    : {t_latent:7.3f}s")
print(f"  predict (label transfer)     : {t_predict:7.3f}s")
print(f"  Total NB elapsed             : {t_total:7.1f}s  ({t_total/60:5.1f} min)")
print()
print("Compare vs Seurat CCA (from NB02 Timing Summary):")
print("  CCA FindTransferAnchors is typically >> scANVI pure inference")
print("  (CCA scales O(n²) with cells; scANVI inference is O(n) / linear)")


=== scvi/02 Timing Summary ===
  Mode                         : pure inference
  get_latent_representation    :   0.467s
  predict (label transfer)     :   0.378s
  Total NB elapsed             :    36.4s  (  0.6 min)

Compare vs Seurat CCA (from NB02 Timing Summary):
  CCA FindTransferAnchors is typically >> scANVI pure inference
  (CCA scales O(n²) with cells; scANVI inference is O(n) / linear)
